📍 GeoGenius – AI-Driven Location Intelligence for Coffee Shop Placement
A RAG + Scoring Engine System for Intelligent Geo-Business Decisions

Project Overview

Opening a new retail outlet—especially a coffee shop—depends heavily on footfall, competition, and rent economics. Traditional decision-making takes days of manual analysis.

GeoGenius is an AI-powered system that provides:

Smart location recommendations

Retrieval-Augmented Geo Insights

LLM-generated business explanations

Multi-city scoring and ranking

This project simulates how real companies like Starbucks, Dunkin, or Blue Tokai evaluate expansion opportunities.

The system works across major Indian metro cities:
Mumbai, Hyderabad, Bengaluru, Delhi-NCR, Andhra Pradesh, and more.

In [ ]:
!pip install -q sentence-transformers scikit-learn transformers torch pandas numpy tqdm


In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame([
    # Maharashtra
    {"zipcode": 400053, "area_name": "Andheri West, Mumbai", "state": "Maharashtra",
     "avg_daily_footfall": 12000, "rent_index": 0.75, "num_coffee_shops": 5, "median_income": 70000},

    {"zipcode": 400049, "area_name": "Bandra West, Mumbai", "state": "Maharashtra",
     "avg_daily_footfall": 18000, "rent_index": 1.00, "num_coffee_shops": 12, "median_income": 110000},

    {"zipcode": 400050, "area_name": "Dadar, Mumbai", "state": "Maharashtra",
     "avg_daily_footfall": 9000, "rent_index": 0.60, "num_coffee_shops": 4, "median_income": 55000},

    {"zipcode": 400069, "area_name": "Andheri East, Mumbai", "state": "Maharashtra",
     "avg_daily_footfall": 8000, "rent_index": 0.50, "num_coffee_shops": 3, "median_income": 50000},

    # Telangana
    {"zipcode": 500081, "area_name": "Hitech City, Hyderabad", "state": "Telangana",
     "avg_daily_footfall": 20000, "rent_index": 0.90, "num_coffee_shops": 15, "median_income": 95000},

    {"zipcode": 500032, "area_name": "Gachibowli, Hyderabad", "state": "Telangana",
     "avg_daily_footfall": 16000, "rent_index": 0.70, "num_coffee_shops": 10, "median_income": 85000},

    {"zipcode": 500072, "area_name": "Kukatpally, Hyderabad", "state": "Telangana",
     "avg_daily_footfall": 11000, "rent_index": 0.55, "num_coffee_shops": 6, "median_income": 60000},

    # Andhra Pradesh
    {"zipcode": 522002, "area_name": "Guntur City Center", "state": "Andhra Pradesh",
     "avg_daily_footfall": 7000, "rent_index": 0.40, "num_coffee_shops": 2, "median_income": 35000},

    {"zipcode": 530016, "area_name": "Visakhapatnam MVP Colony", "state": "Andhra Pradesh",
     "avg_daily_footfall": 14000, "rent_index": 0.65, "num_coffee_shops": 7, "median_income": 65000},

    {"zipcode": 515001, "area_name": "Anantapur Downtown", "state": "Andhra Pradesh",
     "avg_daily_footfall": 6000, "rent_index": 0.30, "num_coffee_shops": 1, "median_income": 28000},

    # Karnataka
    {"zipcode": 560103, "area_name": "Bellandur, Bengaluru", "state": "Karnataka",
     "avg_daily_footfall": 19000, "rent_index": 0.85, "num_coffee_shops": 13, "median_income": 100000},

    {"zipcode": 560034, "area_name": "Koramangala, Bengaluru", "state": "Karnataka",
      "avg_daily_footfall": 17000, "rent_index": 0.80, "num_coffee_shops": 14, "median_income": 95000},

    {"zipcode": 560048, "area_name": "Whitefield, Bengaluru", "state": "Karnataka",
      "avg_daily_footfall": 15000, "rent_index": 0.78, "num_coffee_shops": 9, "median_income": 90000},

    # Tamil Nadu
    {"zipcode": 600042, "area_name": "Velachery, Chennai", "state": "Tamil Nadu",
      "avg_daily_footfall": 13000, "rent_index": 0.58, "num_coffee_shops": 8, "median_income": 62000},

    {"zipcode": 600018, "area_name": "Adyar, Chennai", "state": "Tamil Nadu",
      "avg_daily_footfall": 11000, "rent_index": 0.65, "num_coffee_shops": 7, "median_income": 70000},

    # Delhi NCR
    {"zipcode": 110017, "area_name": "Hauz Khas, Delhi", "state": "Delhi NCR",
      "avg_daily_footfall": 15500, "rent_index": 0.95, "num_coffee_shops": 11, "median_income": 105000},

    {"zipcode": 122002, "area_name": "Cyber City, Gurugram", "state": "Delhi NCR",
      "avg_daily_footfall": 22000, "rent_index": 1.10, "num_coffee_shops": 20, "median_income": 125000},
])

display(df)
print("Total rows:", len(df))


,zipcode,area_name,state,avg_daily_footfall,rent_index,num_coffee_shops,median_income
0,400053,"Andheri West, Mumbai",Maharashtra,12000,0.75,5,70000
1,400049,"Bandra West, Mumbai",Maharashtra,18000,1.00,12,110000
2,400050,"Dadar, Mumbai",Maharashtra,9000,0.60,4,55000
3,400069,"Andheri East, Mumbai",Maharashtra,8000,0.50,3,50000
4,500081,"Hitech City, Hyderabad",Telangana,20000,0.90,15,95000
5,500032,"Gachibowli, Hyderabad",Telangana,16000,0.70,10,85000
6,500072,"Kukatpally, Hyderabad",Telangana,11000,0.55,6,60000
7,522002,Guntur City Center,Andhra Pradesh,7000,0.40,2,35000
8,530016,Visakhapatnam MVP Colony,Andhra Pradesh,14000,0.65,7,65000
9,515001,Anantapur Downtown,Andhra Pradesh,6000,0.30,1,28000


Total rows: 17


In [ ]:
# Convert each dataset row to a clean text document for RAG
def row_to_text(row):
    return (
        f"{row['area_name']} in {row['state']} (zipcode {row['zipcode']}): "
        f"footfall={row['avg_daily_footfall']}, "
        f"coffee_shops={row['num_coffee_shops']}, "
        f"rent_index={row['rent_index']}, "
        f"median_income={row['median_income']}"
    )

# Create docs list
docs = [
    {
        "id": str(r["zipcode"]),
        "text": row_to_text(r),
        "meta": r.to_dict()
    }
    for _, r in df.iterrows()
]

# Verify
print("Docs prepared:", len(docs))
docs[:5]


Docs prepared: 17


[{'id': '400053',
  'text': 'Andheri West, Mumbai in Maharashtra (zipcode 400053): footfall=12000, coffee_shops=5, rent_index=0.75, median_income=70000',
  'meta': {'zipcode': 400053,
   'area_name': 'Andheri West, Mumbai',
   'state': 'Maharashtra',
   'avg_daily_footfall': 12000,
   'rent_index': 0.75,
   'num_coffee_shops': 5,
   'median_income': 70000}},
 {'id': '400049',
  'text': 'Bandra West, Mumbai in Maharashtra (zipcode 400049): footfall=18000, coffee_shops=12, rent_index=1.0, median_income=110000',
  'meta': {'zipcode': 400049,
   'area_name': 'Bandra West, Mumbai',
   'state': 'Maharashtra',
   'avg_daily_footfall': 18000,
   'rent_index': 1.0,
   'num_coffee_shops': 12,
   'median_income': 110000}},
 {'id': '400050',
  'text': 'Dadar, Mumbai in Maharashtra (zipcode 400050): footfall=9000, coffee_shops=4, rent_index=0.6, median_income=55000',
  'meta': {'zipcode': 400050,
   'area_name': 'Dadar, Mumbai',
   'state': 'Maharashtra',
   'avg_daily_footfall': 9000,
   'rent_ind

In [ ]:


from sentence_transformers import SentenceTransformer
import numpy as np
import json, os

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

EMB_PATH = "doc_embeddings_local.npy"
META_PATH = "docs_meta.json"

# Step 1: remove old cache
if os.path.exists(EMB_PATH):
    os.remove(EMB_PATH)
    print("Old embedding cache removed!")

# Step 2: recompute embeddings for ALL docs
texts = [d["text"] for d in docs]
print("Computing embeddings for", len(texts), "documents...")

doc_embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

# Step 3: save updated embeddings + metadata
np.save(EMB_PATH, doc_embeddings)
with open(META_PATH, "w", encoding="utf8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

print("New embeddings created:", doc_embeddings.shape)


Old embedding cache removed!
Computing embeddings for 17 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

New embeddings created: (17, 384)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_local(query, k=3):
    q_emb = embed_model.encode([query])
    sims = cosine_similarity(q_emb, doc_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top_idx]

# FULL detailed retriever test
query = "Best place to open a coffee shop for young professionals"
print("Query:", query)
print("Retrieved (index, similarity):", retrieve_local(query, k=3))

print("\nMatched docs:")
for idx, score in retrieve_local(query, k=3):
    print(f"\n→ Rank {idx} | similarity={score:.4f}")
    print("Doc ID:", docs[idx]["id"])
    print("Text:", docs[idx]["text"])


Query: Best place to open a coffee shop for young professionals
Retrieved (index, similarity): [(15, 0.3706112205982208), (16, 0.3527872562408447), (11, 0.3394598662853241)]

Matched docs:

→ Rank 15 | similarity=0.3706
Doc ID: 110017
Text: Hauz Khas, Delhi in Delhi NCR (zipcode 110017): footfall=15500, coffee_shops=11, rent_index=0.95, median_income=105000

→ Rank 16 | similarity=0.3528
Doc ID: 122002
Text: Cyber City, Gurugram in Delhi NCR (zipcode 122002): footfall=22000, coffee_shops=20, rent_index=1.1, median_income=125000

→ Rank 11 | similarity=0.3395
Doc ID: 560034
Text: Koramangala, Bengaluru in Karnataka (zipcode 560034): footfall=17000, coffee_shops=14, rent_index=0.8, median_income=95000


In [ ]:
import pandas as pd
import numpy as np
from typing import List, Dict, Optional, Tuple

# -----------------------------
# Scoring + Recommendation (robust)
# -----------------------------
DEFAULT_WEIGHTS = {"footfall": 0.6, "competition": -0.3, "rent": -0.1}

PRESET_WEIGHTS = {
    "balanced": DEFAULT_WEIGHTS,
    "low_competition": {"footfall": 0.5, "competition": -0.45, "rent": -0.05},
    "low_rent": {"footfall": 0.55, "competition": -0.25, "rent": -0.20}
}

def score_locations(docs_meta: List[Dict], weights: Optional[Dict]=None) -> pd.DataFrame:
    """
    Compute normalized scores for locations.
    docs_meta: list of metadata dicts containing keys:
      - zipcode, area_name, avg_daily_footfall, num_coffee_shops, rent_index (optional median_income)
    weights: dict with keys 'footfall','competition','rent' (competition and rent should be negative)
    Returns sorted DataFrame with columns: zipcode, area_name, footfall, competition, rent_index, nf, nc, score, confidence
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS

    rows = []
    for d in docs_meta:
        # safe extraction with defaults
        zipcode = int(d.get("zipcode", 0))
        area_name = d.get("area_name", "Unknown")
        foot = float(d.get("avg_daily_footfall", 0))
        comp = float(d.get("num_coffee_shops", 0))
        rent = float(d.get("rent_index", 0))

        rows.append({
            "zipcode": zipcode,
            "area_name": area_name,
            "footfall": foot,
            "competition": comp,
            "rent_index": rent
        })

    df = pd.DataFrame(rows)

    # normalization with safe fallback
    def safe_norm(series):
        rng = np.ptp(series)
        if rng == 0:
            return series / (series.max() or 1)  # all same -> keep ratio
        return (series - series.min()) / rng

    df["nf"] = safe_norm(df["footfall"])               # normalized footfall (0-1)
    # competition: lower is better -> invert after normalization
    df["nc"] = 1.0 - safe_norm(df["competition"])
    # rent: lower is better -> will be applied as negative weight
    df["nrent"] = safe_norm(df["rent_index"])

    df["score"] = (
        weights.get("footfall", 0.0) * df["nf"] +
        weights.get("competition", 0.0) * df["nc"] +
        weights.get("rent", 0.0) * (-df["nrent"])
    )

    # Confidence: rescale scores into 0-1 using min-max of score
    score_min, score_max = df["score"].min(), df["score"].max()
    if score_max - score_min == 0:
        df["confidence"] = 0.5
    else:
        df["confidence"] = (df["score"] - score_min) / (score_max - score_min)

    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    # round values for readability
    df["score"] = df["score"].round(4)
    df["confidence"] = df["confidence"].round(3)
    df[["nf","nc","nrent"]] = df[["nf","nc","nrent"]].round(3)

    return df

def deterministic_recommendation(k: int = 3,
                                 docs_list: Optional[List[Dict]] = None,
                                 weights: Optional[Dict] = None,
                                 preset: Optional[str] = None,
                                 explain: bool = False
                                ) -> Tuple[str, pd.DataFrame]:
    """
    Return top-k recommendation summary and the full score table.
    - If docs_list is None, uses global `docs` variable (list of {"meta":...} or same shape).
    - weights: custom weight dict OR use preset name via `preset` param.
    - explain=True returns an extra short explanation appended to the answer text.
    """
    # get metadata list
    if docs_list is None:
        # docs may be list of {"id","text","meta"}; convert to meta list
        try:
            docs_meta = [d["meta"] for d in docs]
        except Exception:
            # fallback if docs are already meta dicts
            docs_meta = list(docs)
    else:
        docs_meta = docs_list

    # choose weights
    if preset:
        weights = PRESET_WEIGHTS.get(preset, DEFAULT_WEIGHTS)
    if weights is None:
        weights = DEFAULT_WEIGHTS

    df_sc = score_locations(docs_meta, weights=weights)
    top = df_sc.head(k)

    lines = []
    for _, r in top.iterrows():
        lines.append(
            f"{r['area_name']} (zipcode {int(r['zipcode'])}) — score {r['score']:.3f}, "
            f"conf {r['confidence']:.2f} | footfall {int(r['footfall'])}, comp {int(r['competition'])}, rent {r['rent_index']}"
        )

    answer = "Top recommended locations:\n" + "\n".join(lines)
    srcs = top["zipcode"].astype(int).astype(str).tolist()
    answer += "\nSources: [" + ",".join(srcs) + "]"

    if explain:
        # short deterministic explanation (no LLM) — customizable
        top0 = top.iloc[0]
        explanation = (f"The top pick {top0['area_name']} balances high footfall (nf={top0['nf']:.2f}) "
                       f"with acceptable competition and rent, producing the highest combined score.")
        answer = answer + "\n\nExplanation: " + explanation

    return answer, df_sc

# -----------------------------
# Example usage (run to test)
# -----------------------------
# 1) Default recommendation (top 3)
# ans_text, scores_df = deterministic_recommendation(k=3, preset="balanced", explain=True)
# print(ans_text)
# display(scores_df.head(10))

# 2) If you want to emphasize low competition:
# ans2, scores2 = deterministic_recommendation(k=5, preset="low_competition", explain=True)
# print(ans2)
# display(scores2.head(10))


In [ ]:
ans_text, scores_df = deterministic_recommendation(k=3, preset="balanced", explain=True)
print(ans_text)
display(scores_df)


Top recommended locations:
Cyber City, Gurugram (zipcode 122002) — score 0.700, conf 1.00 | footfall 22000, comp 20, rent 1.1
Hitech City, Hyderabad (zipcode 500081) — score 0.521, conf 0.82 | footfall 20000, comp 15, rent 0.9
Bellandur, Bengaluru (zipcode 560103) — score 0.446, conf 0.75 | footfall 19000, comp 13, rent 0.85
Sources: [122002,500081,560103]

Explanation: The top pick Cyber City, Gurugram balances high footfall (nf=1.00) with acceptable competition and rent, producing the highest combined score.


,zipcode,area_name,footfall,competition,rent_index,nf,nc,nrent,score,confidence
0,122002,"Cyber City, Gurugram",22000.0,20.0,1.10,1.000,0.000,1.000,0.7000,1.000
1,500081,"Hitech City, Hyderabad",20000.0,15.0,0.90,0.875,0.263,0.750,0.5211,0.821
2,560103,"Bellandur, Bengaluru",19000.0,13.0,0.85,0.812,0.368,0.688,0.4457,0.746
3,400049,"Bandra West, Mumbai",18000.0,12.0,1.00,0.750,0.421,0.875,0.4112,0.711
4,560034,"Koramangala, Bengaluru",17000.0,14.0,0.80,0.688,0.316,0.625,0.3803,0.680
5,110017,"Hauz Khas, Delhi",15500.0,11.0,0.95,0.594,0.474,0.812,0.2954,0.595
6,500032,"Gachibowli, Hyderabad",16000.0,10.0,0.70,0.625,0.526,0.500,0.2671,0.567
7,560048,"Whitefield, Bengaluru",15000.0,9.0,0.78,0.562,0.579,0.600,0.2238,0.524
8,530016,Visakhapatnam MVP Colony,14000.0,7.0,0.65,0.500,0.684,0.438,0.1385,0.438
9,600042,"Velachery, Chennai",13000.0,8.0,0.58,0.438,0.632,0.350,0.1080,0.408


In [ ]:
# ========== UPDATED RETRIEVER (FINAL) ==========
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_local(query: str, k: int = 3,
                   state_filter: str | None = None,
                   min_footfall: int | None = None,
                   return_docs: bool = True):
    """
    Retrieve top-k most relevant documents for the query.
    Supports:
    - state_filter="Karnataka"
    - min_footfall=15000
    Returns either:
    - list of (doc_dict, similarity)
    - or (index, similarity) if return_docs=False
    """

    if 'embed_model' not in globals() or 'doc_embeddings' not in globals():
        raise RuntimeError("❌ embeddings not found. Run the embedding cell first.")

    # Candidate list
    candidate_indices = list(range(len(docs)))

    # Apply filters
    if state_filter or min_footfall:
        filtered = []
        for i, d in enumerate(docs):
            meta = d.get("meta", {})
            if state_filter:
                if state_filter.lower() not in str(meta.get("state","")).lower():
                    continue
            if min_footfall:
                if meta.get("avg_daily_footfall", 0) < min_footfall:
                    continue
            filtered.append(i)
        candidate_indices = filtered

    if not candidate_indices:
        return []

    # Encode query
    q_emb = embed_model.encode([query])
    cand_embs = doc_embeddings[np.array(candidate_indices)]

    sims = cosine_similarity(q_emb, cand_embs)[0]
    top_local = np.argsort(sims)[::-1][:k]

    results = []
    for pos in top_local:
        global_idx = candidate_indices[int(pos)]
        score = float(sims[int(pos)])
        if return_docs:
            results.append((docs[global_idx], score))
        else:
            results.append((int(global_idx), score))

    return results


# Quick verification
print("Docs count:", len(docs))
test = retrieve_local("Best place to open a coffee shop for young professionals", k=3)
print("\nTop 3 retrieved:")
for doc, sim in test:
    print(f"{doc['id']} | sim={sim:.4f} | {doc['text']}")


Docs count: 17

Top 3 retrieved:
110017 | sim=0.3706 | Hauz Khas, Delhi in Delhi NCR (zipcode 110017): footfall=15500, coffee_shops=11, rent_index=0.95, median_income=105000
122002 | sim=0.3528 | Cyber City, Gurugram in Delhi NCR (zipcode 122002): footfall=22000, coffee_shops=20, rent_index=1.1, median_income=125000
560034 | sim=0.3395 | Koramangala, Bengaluru in Karnataka (zipcode 560034): footfall=17000, coffee_shops=14, rent_index=0.8, median_income=95000


In [ ]:
# ========== UPDATED EXPLANATION GENERATOR ==========

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

GEN_MODEL_NAME = "distilgpt2"   # lightweight & fast for simple summaries

# Load model safely
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME)

# Fix pad token issues for GPT2-like models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1   # CPU
)

def generate_explanation(text_summary, max_new_tokens=50):
    """
    Generates a clean 1–2 sentence explanation for the deterministic ranking.
    The model is instructed not to invent data outside the summary.
    """
    prompt = (
        "You are a helpful analyst. Using only the information inside the summary below, "
        "write a 1–2 sentence explanation of why the top location is recommended. "
        "Do not add new facts.\n\n"
        f"SUMMARY:\n{text_summary}\n\n"
        "EXPLANATION:"
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,       # deterministic output
        temperature=0.0,       # no hallucination
        pad_token_id=tokenizer.pad_token_id
    )

    raw = output[0]["generated_text"]

    # Extract only text AFTER "EXPLANATION:"
    if "EXPLANATION:" in raw:
        result = raw.split("EXPLANATION:", 1)[1].strip()
    else:
        result = raw.strip()

    # Clean excessive repetition or cutoff
    result = result.split("\n")[0].strip()

    return result


Device set to use cpu


In [ ]:
summary_text = ans_text   # from deterministic_recommendation()
print("Summary:\n", summary_text)

exp = generate_explanation(summary_text)
print("\nGenerated Explanation:\n", exp)


Summary:
 Top recommended locations:
Cyber City, Gurugram (zipcode 122002) — score 0.700, conf 1.00 | footfall 22000, comp 20, rent 1.1
Hitech City, Hyderabad (zipcode 500081) — score 0.521, conf 0.82 | footfall 20000, comp 15, rent 0.9
Bellandur, Bengaluru (zipcode 560103) — score 0.446, conf 0.75 | footfall 19000, comp 13, rent 0.85
Sources: [122002,500081,560103]

Explanation: The top pick Cyber City, Gurugram balances high footfall (nf=1.00) with acceptable competition and rent, producing the highest combined score.

Generated Explanation:
 The top pick Cyber City, Gurugram balances high footfall (nf=1.00) with acceptable competition and rent, producing the highest combined score.


In [ ]:
# ===== Improved LLM explanation utilities (replace previous versions) =====
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ensure generator loaded (if not already)
GEN_MODEL_NAME = "distilgpt2"
if 'generator' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)

def _clean_repetition(text: str) -> str:
    """Collapse obvious repeated phrases and trim to first sentence."""
    # collapse repeated substrings (simple heuristic)
    # e.g., "The location is located... The location is located..." -> keep one
    parts = re.split(r'(\.|\?|!)\s*', text)
    if not parts:
        return text.strip()
    # Reconstruct first logical sentence
    # parts is like [sent0, sep0, rest..., ...]
    first_sentence = (parts[0] + (parts[1] if len(parts) > 1 else "")).strip()
    # remove repeated phrase patterns inside first_sentence
    # collapse sequences like "The location is located in the city of Mumbai. The location is located..."
    first_sentence = re.sub(r'\b(.{5,80}?)\b(?:\s+\1\b)+', r'\1', first_sentence)
    # remove duplicate adjacent words (small cleanup)
    first_sentence = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', first_sentence, flags=re.IGNORECASE)
    # final strip and capitalize
    res = first_sentence.strip()
    if len(res) > 0 and res[-1] not in ".!?":
        res = res + "."
    return res

def generate_explanation_for_top_strict(top_meta: dict, max_new_tokens:int=40) -> str:
    """
    Strict LLM explanation for top_meta. Falls back to deterministic explanation if LLM output is invalid.
    top_meta must contain area_name, avg_daily_footfall, num_coffee_shops, rent_index, zipcode.
    """
    summary = (
        f"Top location: {top_meta.get('area_name')} (zipcode {top_meta.get('zipcode')}). "
        f"Footfall={int(top_meta.get('avg_daily_footfall',0))}, "
        f"coffee_shops={int(top_meta.get('num_coffee_shops',0))}, "
        f"rent_index={top_meta.get('rent_index')}."
    )

    prompt = (
        "You are a concise business analyst. Using ONLY the facts in the SUMMARY below, "
        "write ONE short sentence (max) explaining WHY this location is recommended for opening a coffee shop targeting young professionals. "
        "Do NOT add any new facts, and do NOT repeat phrases.\n\n"
        f"SUMMARY:\n{summary}\n\nEXPLANATION:"
    )

    try:
        out = generator(prompt, max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False,
                        pad_token_id=tokenizer.pad_token_id)
        raw = out[0]["generated_text"]
        # Extract text after EXPLANATION:
        if "EXPLANATION:" in raw:
            raw = raw.split("EXPLANATION:",1)[1].strip()
        raw = raw.strip()
        cleaned = _clean_repetition(raw)
        # basic validity checks
        if len(cleaned) < 6 or re.search(r'location is located', cleaned.lower()):
            # fallback deterministic
            raise ValueError("LLM output invalid or repetitive — falling back.")
        return cleaned
    except Exception:
        # Deterministic fallback
        nf = None
        try:
            nf = float(top_meta.get("avg_daily_footfall",0))
        except:
            nf = None
        det = (f"{top_meta.get('area_name')} is recommended because it combines high visitor volume "
               f"(footfall={int(top_meta.get('avg_daily_footfall',0))}) with the best balance of competition and rent among candidates.")
        return det

# Also replace the general generator function used previously
def generate_explanation_strict_from_summary(rec_text: str, max_new_tokens:int=40) -> str:
    """
    Uses the summary text (rec_text) and returns a short single-sentence explanation.
    The function will extract the top area from rec_text and call generate_explanation_for_top_strict.
    """
    # extract first zipcode from rec_text (Sources: [...])
    m = re.search(r"Sources:\s*\[([0-9, ]+)\]", rec_text)
    top_zip = None
    if m:
        list_str = m.group(1).split(",")
        if len(list_str) > 0:
            top_zip = list_str[0].strip()
    # find top_meta
    top_meta = None
    if top_zip:
        for d in docs:
            if str(d["meta"].get("zipcode")) == str(top_zip):
                top_meta = d["meta"]
                break
    if not top_meta:
        # fallback: parse first line area name
        lines = rec_text.splitlines()
        if len(lines) > 0:
            first = lines[0]
            # best-effort: get first area_name from df
            area_guess = first.split("—")[0].strip()
            # try to find by area_name
            for d in docs:
                if area_guess.lower() in d["meta"].get("area_name","").lower():
                    top_meta = d["meta"]
                    break
    if not top_meta:
        # cannot find meta => safe deterministic fallback text
        return "The top location is recommended based on the highest combined score of footfall, competition and rent."
    return generate_explanation_for_top_strict(top_meta, max_new_tokens=max_new_tokens)

# Replace the generate_explanation_for_top and generate_explanation names used elsewhere
generate_explanation_for_top = generate_explanation_for_top_strict
generate_explanation = generate_explanation_strict_from_summary

print("Improved explanation utilities installed.")


Improved explanation utilities installed.


In [ ]:
def geo_answer_llm(
    query: str,
    k: int = 3,
    state: Optional[str] = None,
    min_footfall: Optional[int] = None,
    weights: Optional[Dict] = None,
    preset: Optional[str] = None
):
    # 1. Retrieve relevant documents using the updated retriever
    retrieved_results = retrieve_local(query, k=k*2, state_filter=state, min_footfall=min_footfall, return_docs=True)
    retrieved_docs_meta = [res[0]['meta'] for res in retrieved_results]

    # 2. Score the retrieved documents (or all if no retrieval was done)
    # deterministic_recommendation expects a list of meta dicts
    ans_text, scores_df = deterministic_recommendation(
        k=k,
        docs_list=retrieved_docs_meta if retrieved_docs_meta else docs, # Use all docs if no retrieval or filtered
        weights=weights,
        preset=preset,
        explain=False # LLM will generate explanation
    )

    # 3. Generate LLM-enhanced explanation
    llm_explanation = generate_explanation(ans_text)

    return {
        "answer": ans_text,
        "explanation": llm_explanation,
        "scores_df": scores_df.head(k).to_dict('records'),
        "raw_scores_df": scores_df
    }

# Run the LLM-enhanced answer for the entire 17-row dataset (NO state filter)
out = geo_answer_llm(
    "Where should I open a coffee shop in India?",
    k=5,
    state=None   # <-- IMPORTANT: This forces ALL cities, not just Mumbai
)

print("\nLLM explanation output:\n",
      out.get("explanation", "no explanation field found"))



LLM explanation output:
 Cyber City, Gurugram is recommended because it combines high visitor volume (footfall=22000) with the best balance of competition and rent among candidates.
